### Validate interaction of iNaturalist

Biotic interaction with image data are documented a priori via iNaturalist; however, these annotations are not routinely verified as the research-grade data. Consequently, we do not treat iNaturalist as a primary source of valid biotic interactions. Instead, we retain only interactions that are independently supported by at least one additional resource (a separate curated dataset, other than iNaturalist) indexed in the GloBI database.

### NOTE: source data needs to be downloaded prior:

* **interactions.csv.gz** ([10.5281/zenodo.14640564](https://zenodo.org/records/14640564/files/interactions.csv.gz?download=1)) [2.36 GB]
* **backbone.zip** ([DOI10.15468/39omei](https://www.gbif.org/dataset/d7dddbf4-2cf0-4f39-9b2a-bb099caae36c)) [< 1.0 GB]

See details in: https://anonymous.4open.science/r/biointeract-E615/data_collection_pipeline/README.md

Libraries:

In [ ]:
import pandas as pd
import duckdb
import zipfile
import tempfile
import shutil

Load iNaturalist interactions generated from **interaction_data_collection.ipynb**:

In [ ]:
df = pd.read_parquet("interactions_output_file.parquet")
df

,sourceTaxonName,sourceTaxonRank,targetTaxonName,targetTaxonRank,interactionTypeName,sourceTaxonKingdomName,sourceTaxonPhylumName,sourceTaxonClassName,sourceTaxonOrderName,sourceTaxonFamilyName,...,targetTaxonGenusName,taxonID_y,year,month,day,decimalLatitude,decimalLongitude,identifier,license,referenceCitation
0,Episyrphus balteatus,species,Bellis,genus,visits,Animalia,Arthropoda,Insecta,Diptera,Syrphidae,...,Bellis,3117399,2021,6,7,42.796693,-1.633688,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/82003482
1,Rhaphuma gracilipes,species,Acer platanoides,species,interactsWith,Animalia,Arthropoda,Insecta,Coleoptera,Cerambycidae,...,Acer,3189846,2021,6,7,55.584631,50.566598,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/82003567
2,Rhaphuma gracilipes,species,Acer platanoides,species,interactsWith,Animalia,Arthropoda,Insecta,Coleoptera,Cerambycidae,...,Acer,3189846,2021,6,7,55.584631,50.566598,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/82003567
3,Rhaphuma gracilipes,species,Acer platanoides,species,interactsWith,Animalia,Arthropoda,Insecta,Coleoptera,Cerambycidae,...,Acer,3189846,2021,6,7,55.584631,50.566598,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/82003567
4,Dasineura pellex,species,Fraxinus americana,species,interactsWith,Animalia,Arthropoda,Insecta,Diptera,Cecidomyiidae,...,Fraxinus,3172327,2021,6,7,42.390917,-76.373081,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/82005496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1693611,Hemadas nubilipennis,species,Vaccinium,genus,interactsWith,Animalia,Arthropoda,Insecta,Hymenoptera,Pteromalidae,...,Vaccinium,2882813,2023,5,26,44.847231,-67.156949,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/16403...
1693612,Hemadas nubilipennis,species,Vaccinium,genus,interactsWith,Animalia,Arthropoda,Insecta,Hymenoptera,Pteromalidae,...,Vaccinium,2882813,2023,5,26,44.847231,-67.156949,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/16403...
1693613,Catopsilia pomona,species,Duranta erecta,species,visitsFlowersOf,Animalia,Arthropoda,Insecta,Lepidoptera,Pieridae,...,Duranta,2925655,2023,5,27,18.553805,73.842292,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/16403...
1693614,Austroscaeva occidentalis,species,Coreopsis lanceolata,species,visitsFlowersOf,Animalia,Arthropoda,Insecta,Diptera,Syrphidae,...,Coreopsis,3133938,2021,11,30,-36.833368,-73.031894,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/16403...


Load data with **duckdb** and keep columns **sourceTaxonName**, **sourceTaxonRank**, **targetTaxonName**, **targetTaxonRank**, **referenceCitation**, **interactionTypeName** that specifically **DO NOT** contain link to iNaturalist observations:

In [ ]:
import duckdb
df_globi = duckdb.query("""
    SELECT 
        sourceTaxonName, 
        sourceTaxonRank,
        targetTaxonName, 
        targetTaxonRank,
        referenceCitation,
        interactionTypeName
    FROM 'interactions.csv.gz'
    WHERE referenceCitation NOT LIKE '%www.inaturalist.org/observations%'
""").to_df()
df_globi

,sourceTaxonName,sourceTaxonRank,targetTaxonName,targetTaxonRank,referenceCitation,interactionTypeName
0,Andrena quintilis,species,Pycnanthemum flexuosum,species,"Robertson, C. 1929. Flowers and insects: lists...",visitsFlowersOf
1,Tetrachrysis,genus,Pastinaca sativa,species,"Robertson, C. 1929. Flowers and insects: lists...",visitsFlowersOf
2,Tachytes harpax,species,Pycnanthemum flexuosum,species,"Robertson, C. 1929. Flowers and insects: lists...",visitsFlowersOf
3,Themira putris,species,Sium suave,species,"Robertson, C. 1929. Flowers and insects: lists...",visitsFlowersOf
4,Trirhoptrasema,genus,Sium suave,species,"Robertson, C. 1929. Flowers and insects: lists...",visitsFlowersOf
...,...,...,...,...,...,...
22689992,Toxoplasma gondii,species,Mephitis mephitis,species,"Dubey, J. P., P. G. Parnell, C. Sreekumar, M. ...",endoparasiteOf
22689993,Capillaria procyonis,species,Procyon lotor,species,"Snyder, D. E. 1988a. Indirect Immunofluorescen...",endoparasiteOf
22689994,Trichinella spiralis,species,Procyon lotor,species,"Snyder, D. E. 1988a. Indirect Immunofluorescen...",endoparasiteOf
22689995,Ixodes scapularis,species,Procyon lotor,species,"Slajchert, T., U. D. Kitron, C. J. Jones, and ...",ectoparasiteOf


Load GBIF Backbone Taxonomy from downloaded file **backbone.zip**:

In [ ]:
with zipfile.ZipFile("backbone.zip") as z:
    taxon_path = [name for name in z.namelist() if name.endswith("Taxon.tsv")][0]

    with z.open(taxon_path) as src, tempfile.NamedTemporaryFile(suffix=".tsv") as tmp:
        shutil.copyfileobj(src, tmp)
        tmp.flush()

        df_taxon = duckdb.sql(f"""
            SELECT *
            FROM read_csv(
                '{tmp.name}',
                delim='\t',
                header=True,
                ignore_errors=true
            )
        """).to_df()

df_taxon = df_taxon.dropna(subset=["canonicalName"])
df_taxon = df_taxon[df_taxon["taxonomicStatus"] == "accepted"]
df_taxon = df_taxon[
    ["canonicalName", "kingdom", "phylum", "class", "order", "family", "genus"]
]

canonicalName = df_taxon["canonicalName"].to_list()

print(len(df_globi))

df_globi = df_globi[
    df_globi["sourceTaxonName"].isin(canonicalName)
    & df_globi["targetTaxonName"].isin(canonicalName)
]
print(len(df_globi))

df_globi

22689997
14711162


,sourceTaxonName,sourceTaxonRank,targetTaxonName,targetTaxonRank,referenceCitation,interactionTypeName
0,Andrena quintilis,species,Pycnanthemum flexuosum,species,"Robertson, C. 1929. Flowers and insects: lists...",visitsFlowersOf
1,Tetrachrysis,genus,Pastinaca sativa,species,"Robertson, C. 1929. Flowers and insects: lists...",visitsFlowersOf
2,Tachytes harpax,species,Pycnanthemum flexuosum,species,"Robertson, C. 1929. Flowers and insects: lists...",visitsFlowersOf
3,Themira putris,species,Sium suave,species,"Robertson, C. 1929. Flowers and insects: lists...",visitsFlowersOf
5,Sturmia,genus,Sium suave,species,"Robertson, C. 1929. Flowers and insects: lists...",visitsFlowersOf
...,...,...,...,...,...,...
22689990,Neospora caninum,species,Odocoileus virginianus,species,"Anderson, T., DeJardin, A., Howe, D. K., Dubey...",endoparasiteOf
22689992,Toxoplasma gondii,species,Mephitis mephitis,species,"Dubey, J. P., P. G. Parnell, C. Sreekumar, M. ...",endoparasiteOf
22689994,Trichinella spiralis,species,Procyon lotor,species,"Snyder, D. E. 1988a. Indirect Immunofluorescen...",endoparasiteOf
22689995,Ixodes scapularis,species,Procyon lotor,species,"Slajchert, T., U. D. Kitron, C. J. Jones, and ...",ectoparasiteOf


Keep only iNaturalist interactions (df) that appear in other data sources (df_globi):

In [4]:
df_filtered = df.merge(
    df_globi[
        ["sourceTaxonName", "interactionTypeName", "targetTaxonName"]
    ].drop_duplicates(),
    on=["sourceTaxonName", "interactionTypeName", "targetTaxonName"],
    how="inner"
)
df_filtered

,sourceTaxonName,sourceTaxonRank,targetTaxonName,targetTaxonRank,interactionTypeName,sourceTaxonKingdomName,sourceTaxonPhylumName,sourceTaxonClassName,sourceTaxonOrderName,sourceTaxonFamilyName,...,targetTaxonGenusName,taxonID_y,year,month,day,decimalLatitude,decimalLongitude,identifier,license,referenceCitation
0,Phylloxera glabra,species,Quercus,genus,hasHost,Animalia,Arthropoda,Insecta,Hemiptera,Phylloxeridae,...,Quercus,2877951,2021,6,6,49.244853,-123.185456,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/81984027
1,Phylloxera glabra,species,Quercus,genus,hasHost,Animalia,Arthropoda,Insecta,Hemiptera,Phylloxeridae,...,Quercus,2877951,2021,6,6,49.244853,-123.185456,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/81984027
2,Phylloxera glabra,species,Quercus,genus,hasHost,Animalia,Arthropoda,Insecta,Hemiptera,Phylloxeridae,...,Quercus,2877951,2021,6,6,49.244853,-123.185456,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/81984027
3,Phylloxera glabra,species,Quercus,genus,hasHost,Animalia,Arthropoda,Insecta,Hemiptera,Phylloxeridae,...,Quercus,2877951,2021,6,6,49.244853,-123.185456,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/81984027
4,Phylloxera glabra,species,Quercus,genus,hasHost,Animalia,Arthropoda,Insecta,Hemiptera,Phylloxeridae,...,Quercus,2877951,2021,6,6,49.244853,-123.185456,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/81984027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256753,Apis mellifera,species,Taraxacum,genus,visitsFlowersOf,Animalia,Arthropoda,Insecta,Hymenoptera,Apidae,...,Taraxacum,7787708,2023,5,26,44.47299,-77.30793,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/16402...
256754,Hemadas nubilipennis,species,Vaccinium,genus,interactsWith,Animalia,Arthropoda,Insecta,Hymenoptera,Pteromalidae,...,Vaccinium,2882813,2023,5,26,44.847128,-67.156998,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/16403...
256755,Hemadas nubilipennis,species,Vaccinium,genus,interactsWith,Animalia,Arthropoda,Insecta,Hymenoptera,Pteromalidae,...,Vaccinium,2882813,2023,5,26,44.847128,-67.156998,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/16403...
256756,Hemadas nubilipennis,species,Vaccinium,genus,interactsWith,Animalia,Arthropoda,Insecta,Hymenoptera,Pteromalidae,...,Vaccinium,2882813,2023,5,26,44.847231,-67.156949,https://inaturalist-open-data.s3.amazonaws.com...,http://creativecommons.org/licenses/by-nc/4.0/,https://www.inaturalist.org/observations/16403...


Unique interactions in filtered dataset:

In [5]:
grouped_inat = df_filtered.groupby(['sourceTaxonName','interactionTypeName','targetTaxonName']).size().reset_index(name='count')
grouped_inat = grouped_inat.sort_values(
    by=["count"],
    ascending=[False]
)
grouped_inat

,sourceTaxonName,interactionTypeName,targetTaxonName,count
4220,Bombus impatiens,visitsFlowersOf,Solidago,4596
3651,Bombus griseocollis,visitsFlowersOf,Echinacea purpurea,2181
7148,Danaus plexippus,interactsWith,Asclepias syriaca,2005
2213,Apis mellifera,visitsFlowersOf,Solidago,1479
9725,Istocheta aldrichi,parasiteOf,Popillia japonica,1474
...,...,...,...,...
7397,Diaspis manzanitae,interactsWith,Arctostaphylos glandulosa,1
7394,Diaspis echinocacti,interactsWith,Cactaceae,1
7374,Dialictus,visitsFlowersOf,Sanguinaria canadensis,1
7370,Dialictus,interactsWith,Plantaginaceae,1


Total number of interactions after validation with other dataset resources:

In [6]:
len(df_filtered)

256758

Unique interactions in original GLOBI dataset:

In [7]:
grouped_inat = df.groupby(['sourceTaxonName','interactionTypeName','targetTaxonName']).size().reset_index(name='count')
grouped_inat = grouped_inat.sort_values(
    by=["count"],
    ascending=[False]
)
grouped_inat

,sourceTaxonName,interactionTypeName,targetTaxonName,count
44280,Bombus impatiens,visitsFlowersOf,Solidago,4596
8941,Amphibolips quercuspomiformis,interactsWith,Quercus agrifolia,2783
40669,Bombus griseocollis,visitsFlowersOf,Echinacea purpurea,2181
147205,Megacyllene robiniae,visitsFlowersOf,Solidago,2078
44331,Bombus impatiens,visitsFlowersOf,Symphyotrichum,2028
...,...,...,...,...
184827,Platycryptus undatus,visits,Acer rubrum,1
125923,Icaricia saepiolus,visits,Saussurea americana,1
64544,Cercyonis pegala,visits,Apocynum cannabinum,1
64543,Cercyonis pegala,visits,Amorpha canescens,1


Save filtered GLOBI interactions to parquet:

In [ ]:
df_filtered.to_parquet("interactions_output_file.parquet", index=False)